# Lamplighter — catch a real bug before you run

An agent (or a colleague, or you at 2am) hands you a training setup. It looks
fine. It even runs. This notebook shows Lamplighter reading your **actual data**
against your **actual model** and finding two problems that no linter, type
checker, or unit test would catch — *before* you spend the epoch.

The bug here is not planted. `torchvision`'s EMNIST **`letters`** split returns
labels in **1…26**, not 0…25 — a real, long-standing gotcha. Feed those to a
26-output head with `CrossEntropyLoss` and PyTorch asserts and dies mid-epoch
with an opaque CUDA error. Lamplighter says so in plain English at the door.


## 1. Load EMNIST `letters` — as anyone would

Flatten to 784, subsample for a snappy CPU demo. Note we do **nothing** unusual;
this is the obvious way to load it.


In [ ]:
import torch
from torchvision import datasets, transforms

emnist = datasets.EMNIST(root='./data', split='letters', train=True,
                         download=True, transform=transforms.ToTensor())

# A 12,001-sample subset — the odd count matters later.
idx = torch.randperm(len(emnist))[:12001]
X = torch.stack([emnist[i][0].view(-1) for i in idx])   # (12001, 784)
y = torch.tensor([emnist[i][1] for i in idx])           # the labels, as-is

print('X', tuple(X.shape), '| y range', int(y.min()), '…', int(y.max()))


`y` runs **1…26**. Easy to miss — the shapes are right, the dtype is right, and
the split is literally named `letters`, so 26 classes feels correct.


## 2. Register the data and open the editor

`sess.data(...)` hands Lamplighter *references* — nothing is copied, and the
editor's data pickers open already populated.


In [ ]:
import lamplighter

sess = lamplighter.Lamplighter()   # a server in this kernel — no browser yet
sess.data(X=X, y=y)                # references, not copies
sess.open()                        # open the editor


In the app: **New project ▾ → MLP**, then on the canvas set the last **Linear**
node's *Out Features* to **26** (26 letter classes), and on the **Training** tab
pick **CrossEntropyLoss**. Pick `X` and `y` on the dataset node.

The **Pre-flight** panel beside ▶ Run now reads — against your real tensors:

> ✓ &nbsp;`X` — 12001 samples of (784) match the Input  
> ✗ &nbsp;**`y` has classes 1…26 but the model outputs 26** — this would crash
> mid-run — adjust the last layer's out_features

That `1…26` is not a heuristic. It's the actual min and max of the tensor you
registered. ▶ Run stays disabled until it's real.


## 3. See it from the notebook too

The same checks are a function call — the app is just their display surface.
Here's the exact verdict the panel shows, computed on your data:


In [ ]:
from lamplighter.backend.diagnose import diagnose
from lamplighter.backend.templates import TEMPLATES
from lamplighter.backend.schema import DataNode, ModelLink

# The project you built in the app, in code (MLP head → 26, CrossEntropyLoss):
project = TEMPLATES['mlp'].build()
for n in project.models[0].graph.nodes:
    if n.type == 'Linear':
        n.params['out_features'] = 26
project.training = {'loss': 'CrossEntropyLoss'}
project.data_nodes = [DataNode(id='d', kind='dataset', name='D',
    config={'source': 'memory', 'x_var': 'X', 'y_var': 'y', 'batch_size': 40})]
project.links = [ModelLink(id='l', source_data='d', target_model=project.models[0].id)]

for c in diagnose(project, {'X': X, 'y': y}):
    mark = {'ok': '✓', 'warn': '⚠', 'error': '✗'}[c['level']]
    print(f"{mark} {c['title']}")
    if c['detail']:
        print(f"    {c['detail']}")


## 4. The fix, and the run

Subtract one from the labels (0…25) — or, if you prefer, keep them and give the
head 27 outputs. Either way the pre-flight flips green and ▶ Run un-blocks.


In [ ]:
y = y - 1                          # 1…26  →  0…25
sess.data(y=y)                     # repoint — the picker updates live

for c in diagnose(project, {'X': X, 'y': y}):
    mark = {'ok': '✓', 'warn': '⚠', 'error': '✗'}[c['level']]
    print(f"{mark} {c['title']}")


All green. Press **▶ Run** in the app and the curves stream live; the trained
model is `sess.model`, the history `sess.history`.

---

**That's the whole idea.** An LLM writes you a good training loop. What it can't
do is look at your data. Lamplighter does — before the run, against the real
tensors, in plain English.
